In [ ]:
import os

import requests

api_key = os.getenv("OPENROUTER_API_KEY")

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "stepfun/step-3.5-flash:free",
        "messages": [
            {
                "role": "user",
                "content": 'Оцени ответ. Верни строго JSON: {"score": 4, "reasoning": "тест", "key_mistakes": []}',
            }
        ],
        "temperature": 0.1,
        "max_tokens": 200,
    },
    timeout=120,
)
print(response.status_code)
print(response.json()["choices"][0]["message"]["content"])

200
None


In [5]:
import json
import os

import requests

api_key = os.getenv("OPENROUTER_API_KEY")

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "stepfun/step-3.5-flash:free",
        "messages": [
            {
                "role": "user",
                "content": 'Оцени ответ. Верни строго JSON: {"score": 4, "reasoning": "тест", "key_mistakes": []}',
            }
        ],
        "temperature": 0.1,
        "max_tokens": 200,
    },
    timeout=120,
)

# Печатаем весь ответ целиком чтобы понять структуру
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

{
  "id": "gen-1773411296-TDnvblSIb2sHJWd10Ucp",
  "object": "chat.completion",
  "created": 1773411296,
  "model": "stepfun/step-3.5-flash:free",
  "provider": "StepFun",
  "system_fingerprint": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "length",
      "native_finish_reason": "length",
      "message": {
        "role": "assistant",
        "content": null,
        "refusal": null,
        "reasoning": "Хм, пользователь просит оценить ответ в строгом JSON-формате с конкретными полями. Нужно понять, что именно оценивать — вероятно, какой-то предыдущий ответ, но в запросе его нет. \n\nПользователь явно ожидает структурированный ответ, поэтому важно следовать формату. Поскольку исходного ответа для оценки нет, логично предположить, что нужно оценить сам запрос пользователя или его корректность. \n\nЗапрос пользователя грамматически корректен, но требует уточнения объекта оценки. Можно поставить низкий балл из-за отсутствия контекста, но о

In [8]:
"""
Тест API OpenRouter с реальным примером доменной оценки.
Запуск: python test_judge.py
"""

import json
import os
import re

import requests

api_key = os.getenv("OPENROUTER_API_KEY")

# ── Реальный пример из domain_questions.json ──────────────────────────────

QUESTION = (
    "Что такое механизм внимания (attention) в трансформерах и зачем он нужен?"
)

REFERENCE_ANSWER = (
    "Механизм внимания позволяет модели при обработке каждого токена динамически "
    "взвешивать важность всех остальных токенов в последовательности. Это решает "
    "проблему RNN — невозможность эффективно передавать информацию через длинные "
    "зависимости. Внимание вычисляется через матрицы Query, Key и Value: "
    "score = softmax(QK^T / sqrt(d_k)) * V."
)

# Намеренно неполный ответ модели — чтобы судья нашёл что покритиковать
MODEL_ANSWER = (
    "Механизм внимания — это способ модели смотреть на другие слова в предложении "
    "при обработке текущего слова. Это помогает лучше понимать контекст. "
    "Используется в трансформерах вместо рекуррентных сетей."
)

SYSTEM_PROMPT = """Ты — эксперт-оценщик ответов языковых моделей. 
Твоя задача: оценить ответ языковой модели на вопрос по теме архитектуры и обучения LLM.

Оценивай по следующим критериям:
1. Техническая точность (правильность фактов, формул, концепций)
2. Полнота (все ли ключевые аспекты раскрыты)
3. Ясность изложения (структурированность, понятность)

Формат ответа — строго JSON:
{
  "score": <число от 1 до 5>,
  "reasoning": "<краткое обоснование на русском языке, 2-3 предложения>",
  "key_mistakes": ["<ошибка 1>", "<ошибка 2>"]
}

Шкала:
5 — Отличный ответ: всё верно, полно, ясно
4 — Хороший ответ: верно, но неполно или недостаточно ясно
3 — Удовлетворительный: частично верно, есть пробелы
2 — Слабый ответ: есть существенные ошибки
1 — Неверный или нерелевантный ответ"""

USER_PROMPT = f"""**Вопрос:**
{QUESTION}

**Ответ модели:**
{MODEL_ANSWER}

**Эталонный ответ (для справки):**
{REFERENCE_ANSWER}

Оцени ответ модели."""

# ── Запрос ────────────────────────────────────────────────────────────────

print("=" * 60)
print("Sending request to OpenRouter...")
print("Model: stepfun/step-3.5-flash:free")
print(f"Prompt tokens (approx): {len(USER_PROMPT.split())}")
print("=" * 60)

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "stepfun/step-3.5-flash:free",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ],
        "temperature": 0.1,
        "max_tokens": 1500,
        "reasoning": {
            "exclude": True  # убираем reasoning из ответа
        },
    },
    timeout=120,
)

# ── Разбор ответа ─────────────────────────────────────────────────────────

print(f"\nStatus code: {response.status_code}")

raw = response.json()

# Статистика токенов
usage = raw.get("usage", {})
print("\nToken usage:")
print(f"  Prompt tokens:    {usage.get('prompt_tokens', '?')}")
print(
    f"  Reasoning tokens: {usage.get('completion_tokens_details', {}).get('reasoning_tokens', '?')}"
)
print(f"  Completion tokens:{usage.get('completion_tokens', '?')}")
print(f"  Total:            {usage.get('total_tokens', '?')}")
print(f"  Cost:             ${usage.get('cost', 0)}")

# finish_reason — важно проверить что не 'length'
choice = raw["choices"][0]
print(f"\nFinish reason: {choice['finish_reason']}")

# Извлекаем контент с fallback на reasoning
message = choice["message"]
content = message.get("content") or message.get("reasoning", "")

print(f"\nRaw content:\n{content}")

# Парсим JSON из ответа
print("\n" + "=" * 60)
json_match = re.search(r"\{.*\}", content, re.DOTALL)
if json_match:
    try:
        verdict = json.loads(json_match.group())
        print("✅ JSON parsed successfully!")
        print(f"  Score:        {verdict.get('score')}/5")
        print(f"  Reasoning:    {verdict.get('reasoning')}")
        print(f"  Key mistakes: {verdict.get('key_mistakes')}")
    except json.JSONDecodeError as e:
        print(f"❌ JSON parse error: {e}")
        print(f"   Raw match: {json_match.group()}")
else:
    print("❌ No JSON found in response")
    print("   Full response for debugging:")
    print(json.dumps(raw, ensure_ascii=False, indent=2))

Sending request to OpenRouter...
Model: stepfun/step-3.5-flash:free
Prompt tokens (approx): 91

Status code: 200

Token usage:
  Prompt tokens:    464
  Reasoning tokens: 844
  Completion tokens:460
  Total:            924
  Cost:             $0

Finish reason: stop

Raw content:
{
  "score": 3,
  "reasoning": "Ответ верен в общих чертах, но крайне неполон и не раскрывает ключевых технических деталей механизма внимания. Отсутствует объяснение динамического взвешивания, роли матриц Query/Key/Value и формулы вычисления, что делает ответ поверхностным.",
  "key_mistakes": [
    "Не объяснён принцип динамического взвешивания важности токенов",
    "Не упомянуты матрицы Query, Key, Value и их роль",
    "Не показано, как внимание решает проблему длинных зависимостей в RNN"
  ]
}

✅ JSON parsed successfully!
  Score:        3/5
  Reasoning:    Ответ верен в общих чертах, но крайне неполон и не раскрывает ключевых технических деталей механизма внимания. Отсутствует объяснение динамического вз

In [6]:
import sys

os.chdir("/home/danil-pc/fine-tuning_project")
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from eval.perplexity import load_val_data

data = load_val_data("data/final/val.jsonl")
print(f"Loaded {len(data)} samples")
print("First sample keys:", data[0].keys())

Loaded 123 samples
First sample keys: dict_keys(['messages'])


In [7]:
import json
import sys

os.chdir("/home/danil-pc/fine-tuning_project")
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from eval.coding_eval import _run_unit_tests

with open("data/eval/coding_tasks.json") as f:
    tasks = json.load(f)

# Тестируем на заведомо правильном решении
correct_code = """
def flatten(lst):
    result = []
    for item in lst:
        if isinstance(item, list):
            result.extend(flatten(item))
        else:
            result.append(item)
    return result
"""
result = _run_unit_tests(correct_code, tasks[0]["test_code"], tasks[0]["id"])
print("Correct code test:", result)

# Тестируем на заведомо неправильном решении
wrong_code = "def flatten(lst): return lst"
result = _run_unit_tests(wrong_code, tasks[0]["test_code"], tasks[0]["id"])
print("Wrong code test:", result)

Correct code test: {'passed': True, 'error': None, 'error_type': None}
Wrong code test: {'passed': False, 'error': 'AssertionError: Traceback (most recent call last):\n  File "/home/danil-pc/fine-tuning_project/scripts/eval/coding_eval.py", line 127, in _run_unit_tests\n    result = run_tests(namespace[fn_name])\n             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "<string>", line 2, in run_tests\nAssertionError\n', 'error_type': 'AssertionError'}
